# 🎯 Logistic Regression — Binary Classification on Titanic

**Dataset:** [Titanic — Machine Learning from Disaster](https://www.kaggle.com/competitions/titanic)  
**Goal:** Predict whether a passenger survived (1) or died (0)  

---

## What This Notebook Covers

1. **Sigmoid function** — how logistic regression converts any number into a probability
2. **Preprocessing** — cleaning and encoding Titanic data for classification
3. **Baseline** — what a dumb model scores before we build anything
4. **Confusion matrix** — TP, TN, FP, FN explained
5. **Classification metrics** — Precision, Recall, F1, ROC-AUC
6. **Threshold tuning** — the precision-recall tradeoff in practice
7. **Feature importance** — which features drive survival probability
8. **Cross validation** — reliable model evaluation
9. **Full pipeline** — everything combined

---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## 2. The Sigmoid Function

Logistic regression uses the sigmoid function to convert any number into a probability between 0 and 1:

```
σ(z) = 1 / (1 + e^(-z))
```

- z very negative → probability ≈ 0 (predict died)
- z = 0 → probability = 0.5 (uncertain)
- z very positive → probability ≈ 1 (predict survived)

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-10, 10, 300)
prob = sigmoid(z)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(z, prob, color='steelblue', linewidth=3)
ax.axhline(y=0.5, color='red', linestyle='--', linewidth=1.5, label='Decision threshold (0.5)')
ax.axvline(x=0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.annotate('Predict Survived', xy=(5, 0.95), fontsize=11, color='green', fontweight='bold')
ax.annotate('Predict Died', xy=(-9, 0.05), fontsize=11, color='red', fontweight='bold')
ax.fill_between(z, prob, 0.5, where=(prob >= 0.5), alpha=0.1, color='green')
ax.fill_between(z, prob, 0.5, where=(prob < 0.5), alpha=0.1, color='red')
ax.set_title('The Sigmoid Function — Turning Any Number Into a Probability',
             fontsize=14, fontweight='bold')
ax.set_xlabel('z (linear combination of features)', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

# Key sigmoid values
print(f'sigmoid(2.5)  = {sigmoid(2.5):.4f}  → {sigmoid(2.5)*100:.1f}% chance of survival')
print(f'sigmoid(-1.0) = {sigmoid(-1.0):.4f}  → {sigmoid(-1.0)*100:.1f}% chance of survival')
print(f'sigmoid(0)    = {sigmoid(0):.4f}  → exactly 50/50')

## 3. Data Preprocessing

In [ ]:
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Fill missing values
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Drop useless columns
df.drop(columns=['Cabin', 'Name', 'Ticket', 'PassengerId'], inplace=True)

# Feature engineering
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df['FarePerPerson'] = df['Fare'] / df['FamilySize']
df['AgeGroup'] = pd.cut(df['Age'],
                         bins=[0, 12, 18, 35, 60, 100],
                         labels=[1, 2, 3, 4, 5]).astype(int)

# Encode Sex
df['Sex_encoded'] = (df['Sex'] == 'female').astype(int)

# One-hot encode Embarked
embarked_dummies = pd.get_dummies(df['Embarked'], prefix='Embarked', drop_first=True)
df = pd.concat([df, embarked_dummies], axis=1)

# Drop original columns
df.drop(columns=['Sex', 'Embarked'], inplace=True)

# Define features and target
X = df.drop('Survived', axis=1)
y = df['Survived']

print(f'X shape: {X.shape}')
print(f'Missing values: {X.isnull().sum().sum()}')
print(f'Features: {list(X.columns)}')

## 4. Baseline & Model Training

Before training any model, establish what a dumb model would score. The majority class classifier always predicts the most common class — our model must beat this to be useful.

In [ ]:
# Baseline
baseline_accuracy = y.value_counts(normalize=True).max()
print(f'Class distribution:\n{y.value_counts()}\n')
print(f'Majority class baseline accuracy: {baseline_accuracy:.1%}')
print(f'Our model must beat {baseline_accuracy:.1%} to be useful')

In [ ]:
# Train/test split — stratify keeps class ratio balanced
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Pipeline: scale then fit
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42, max_iter=1000))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
print(f'Model accuracy:    {accuracy:.1%}')
print(f'Baseline accuracy: {baseline_accuracy:.1%}')
print(f'Improvement:       +{(accuracy - baseline_accuracy):.1%}')

## 5. Confusion Matrix & Classification Metrics

Accuracy alone is misleading for imbalanced classes. The confusion matrix breaks down predictions into four categories:

| | Predicted Died | Predicted Survived |
|---|---|---|
| **Actually Died** | TN ✅ | FP ❌ |
| **Actually Survived** | FN ❌ | TP ✅ |

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Predicted Died', 'Predicted Survived'],
            yticklabels=['Actually Died', 'Actually Survived'])
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Actual', fontsize=12)
ax.set_xlabel('Predicted', fontsize=12)
plt.tight_layout()
plt.show()

print(f'True Negatives  (correctly predicted died):          {tn}')
print(f'False Positives (predicted survived, actually died): {fp}')
print(f'False Negatives (predicted died, actually survived): {fn}')
print(f'True Positives  (correctly predicted survived):      {tp}')

In [ ]:
# Manual metric calculations
precision_manual = tp / (tp + fp)
recall_manual    = tp / (tp + fn)
accuracy_manual  = (tp + tn) / (tp + tn + fp + fn)

print('Manual calculations:')
print(f'  Precision = {tp} / ({tp} + {fp}) = {precision_manual:.3f}')
print(f'  Recall    = {tp} / ({tp} + {fn}) = {recall_manual:.3f}')
print(f'  Accuracy  = ({tp} + {tn}) / {tp+tn+fp+fn} = {accuracy_manual:.3f}')
print()
print('Sklearn verification:')
print(f'  Precision: {precision_score(y_test, y_pred):.3f}')
print(f'  Recall:    {recall_score(y_test, y_pred):.3f}')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred):.3f}')
print()
# In a lifeboat context, FP (told they have a seat but don't) is worse than FN
# because the person stops looking for safety, putting their life at greater risk
print('Worst error type: False Positive')
print('Reason: telling someone they have a lifeboat seat when they do not')
print('causes them to stop seeking safety — a potentially fatal mistake')

In [ ]:
# Full classification report
print(classification_report(y_test, y_pred, target_names=['Died', 'Survived']))

## 6. ROC Curve & AUC

The ROC curve shows model performance at **every possible threshold**, not just 0.5. AUC (Area Under Curve) summarizes this into one number — closer to 1.0 is better, 0.5 is random guessing.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='steelblue', linewidth=2.5,
        label=f'Logistic Regression (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--',
        linewidth=1.5, label='Random classifier (AUC = 0.5)')
ax.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
ax.set_title('ROC Curve — Logistic Regression on Titanic',
             fontsize=14, fontweight='bold')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f'AUC: {auc:.3f} — model separates survivors from non-survivors {auc*100:.1f}% of the time')

## 7. Threshold Tuning — Precision-Recall Tradeoff

The default threshold of 0.5 is not always optimal. Lowering it increases Recall (catch more survivors) at the cost of Precision. Raising it does the opposite.

In [ ]:
# Metrics at different thresholds
thresholds_to_try = [0.3, 0.4, 0.5, 0.6, 0.7]
results = []
for thresh in thresholds_to_try:
    y_pred_thresh = (y_prob >= thresh).astype(int)
    results.append({
        'Threshold': thresh,
        'Accuracy':  accuracy_score(y_test, y_pred_thresh),
        'Precision': precision_score(y_test, y_pred_thresh, zero_division=0),
        'Recall':    recall_score(y_test, y_pred_thresh, zero_division=0),
        'F1':        f1_score(y_test, y_pred_thresh, zero_division=0)
    })

results_df = pd.DataFrame(results).set_index('Threshold').round(3)
print(results_df)

In [ ]:
# Precision, Recall, F1 vs Threshold — full curve
thresholds_range = np.arange(0.1, 0.9, 0.05)
precisions, recalls, f1s = [], [], []

for thresh in thresholds_range:
    y_pred_t = (y_prob >= thresh).astype(int)
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds_range, precisions, color='#2196F3', linewidth=2, label='Precision')
ax.plot(thresholds_range, recalls,    color='#F44336', linewidth=2, label='Recall')
ax.plot(thresholds_range, f1s,        color='#4CAF50', linewidth=2, label='F1 Score')
ax.axvline(x=0.5, color='gray', linestyle='--', linewidth=1.5, label='Default threshold (0.5)')

best_f1_thresh = thresholds_range[np.argmax(f1s)]
ax.axvline(x=best_f1_thresh, color='#4CAF50', linestyle=':', linewidth=1.5,
           label=f'Best F1 threshold ({best_f1_thresh:.2f})')

ax.set_title('Precision, Recall and F1 vs Decision Threshold', fontsize=14, fontweight='bold')
ax.set_xlabel('Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f'Best threshold for F1: {best_f1_thresh:.2f}')
print(f'F1 at best threshold:  {max(f1s):.3f}')
print(f'F1 at default (0.5):   {f1_score(y_test, y_pred):.3f}')

## 8. Feature Importance

After scaling, the magnitude of each coefficient shows how strongly that feature influences the survival probability. Positive = increases survival probability, negative = decreases it.

In [ ]:
model = pipeline.named_steps['model']
coef_df = pd.DataFrame({
    'Feature': X.columns.tolist(),
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#4CAF50' if c > 0 else '#F44336' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Feature Coefficients\n(green = increases survival probability, red = decreases it)',
             fontweight='bold')
ax.set_xlabel('Coefficient (after scaling)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print('Top features increasing survival:')
print(coef_df.tail(3)[['Feature', 'Coefficient']].to_string(index=False))
print('\nTop features decreasing survival:')
print(coef_df.head(3)[['Feature', 'Coefficient']].to_string(index=False))
print()
# Sex_encoded (female=1) is the strongest positive predictor — matches history
# Pclass is strongly negative — 3rd class had far less access to lifeboats
# This aligns with the historical 'women and children first' evacuation policy

## 9. Cross Validation

In [ ]:
cv_accuracy = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
cv_f1       = cross_val_score(pipeline, X, y, cv=5, scoring='f1')
cv_auc      = cross_val_score(pipeline, X, y, cv=5, scoring='roc_auc')

print('5-Fold Cross Validation Results:')
print(f'  Accuracy: {cv_accuracy.mean():.3f} ± {cv_accuracy.std():.3f}')
print(f'  F1 Score: {cv_f1.mean():.3f} ± {cv_f1.std():.3f}')
print(f'  ROC-AUC:  {cv_auc.mean():.3f} ± {cv_auc.std():.3f}')
print()
print('Low std = consistent model across different data splits')

## 10. Results Summary

| Metric | Score |
|--------|-------|
| Baseline Accuracy | ~61.5% |
| Model Accuracy | ~82% |
| F1 Score | ~0.77 |
| ROC-AUC | ~0.88 |

**Key findings:**
- The model significantly outperforms the majority class baseline (+20%)
- Sex is the strongest predictor — female passengers had dramatically higher survival rates, consistent with the 'women and children first' evacuation policy
- Passenger class (Pclass) is the strongest negative predictor — 3rd class passengers had much lower survival rates due to restricted lifeboat access
- The model struggles most with male survivors — the confusion matrix shows more False Negatives than False Positives, meaning it under-predicts male survival
- Next step: try a non-linear classifier (Decision Tree, Random Forest) which can capture interactions between features that logistic regression cannot

---
**Next:** Support Vector Machines (SVM) — a different approach to finding the optimal decision boundary